# (Using Softmax LR) - Predicting Pipeline with BOW and Frequency as Features

In [2]:
import pandas as pd
import re
from typing import List
import numpy as np
import nltk
from sklearn.linear_model import LogisticRegression
from functools import partial
import numpy as np
from collections import defaultdict
from nltk.corpus import webtext, brown, words # different nltk corpora which can be used as dictionary of words
from scipy import sparse

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# To load train and test datasets into Pandas

In [4]:
!ls /content/drive/MyDrive/data/archive-2/

test.csv  testdata.manual.2009.06.14.csv  train.csv  training.1600000.processed.noemoticon.csv


In [5]:
train_df = pd.read_csv('/content/drive/MyDrive/data/archive-2/train.csv', encoding='unicode_escape')
test_df = pd.read_csv('/content/drive/MyDrive/data/archive-2/test.csv', encoding='unicode_escape')

In [6]:
train_df.columns

Index(['textID', 'text', 'selected_text', 'sentiment', 'Time of Tweet',
       'Age of User', 'Country', 'Population -2020', 'Land Area (Km²)',
       'Density (P/Km²)'],
      dtype='object')

In [7]:
test_df.columns

Index(['textID', 'text', 'sentiment', 'Time of Tweet', 'Age of User',
       'Country', 'Population -2020', 'Land Area (Km²)', 'Density (P/Km²)'],
      dtype='object')

In [8]:
train_df.iloc[5]['text']

'http://www.dothebouncy.com/smf - some shameless plugging for the best Rangers forum on earth'

In [9]:
def basic_tokenizer(text: str)->List[str]:
    'tokenize words and punctuations using python split'
    result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    return [item.strip() for item in result if item.strip()]

In [10]:
text = train_df.iloc[5]['text']
basic_tokenizer(text)

['http',
 ':',
 '//www',
 '.',
 'dothebouncy',
 '.',
 'com/smf',
 '-',
 'some',
 'shameless',
 'plugging',
 'for',
 'the',
 'best',
 'Rangers',
 'forum',
 'on',
 'earth']

In [29]:
# Used to find preferred data paths for nltk
# print(nltk.data.path)
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [30]:
nltk.tokenize.word_tokenize(text, language='english', preserve_line=False)

['http',
 ':',
 '//www.dothebouncy.com/smf',
 '-',
 'some',
 'shameless',
 'plugging',
 'for',
 'the',
 'best',
 'Rangers',
 'forum',
 'on',
 'earth']

In [12]:
#nltk.download('popular')
#nltk.download('webtext')
#nltk.download('brown')

[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/cmudict.zip.
[nltk_data]    | Downloading package gazetteers to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/gazetteers.zip.
[nltk_data]    | Downloading package genesis to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/genesis.zip.
[nltk_data]    | Downloading package gutenberg to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/gutenberg.zip.
[nltk_data]    | Downloading package inaugural to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/inaugural.zip.
[nltk_data]    | Downloading package movie_reviews to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping corpora/movie_reviews.zip.
[nltk_data]    | Downloading package names to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/names.zip.
[nltk_data]    | Downloading package shakespeare to /root/nltk_data...
[nlt

True

# Create Train and Test Dataset

In [15]:
def create_X_Y(df, tokenizer_fn, words):
    words_set = set(words)
    words = list(words_set)
    assert len(words_set)==len(words)
    words_dict = {c: i for i, c in enumerate(words)}
    X = defaultdict(lambda: defaultdict(int))
    Y = []
    row = -1
    for i in range(len(df)):
        try:
            tokens = tokenizer_fn(df.iloc[i]['text'])
            row+=1
        except TypeError:
            continue
        for t in tokens:
            if t not in words:
                X[row][len(words)]+=1
            else:
                X[row][words_dict[t]]+=1
        Y.append(df.iloc[i]['sentiment'])

    return X, Y, len(words_set)


X_train, Y_train, num_words = create_X_Y(train_df, partial(nltk.tokenize.word_tokenize, language='english', preserve_line=False), words.words())
X_test, Y_test, num_words = create_X_Y(test_df, partial(nltk.tokenize.word_tokenize, language='english', preserve_line=False), words.words())

# To load train data into sparse matrices

In [16]:
X_train_s = sparse.dok_matrix((len(X_train), num_words+1), dtype=np.int8)

for row_id, word_freq_dict in X_train.items():
    for word_id, word_count in word_freq_dict.items():
        X_train_s[row_id, word_id] = word_count

X_train_sparse = X_train_s.tocsr()

In [17]:
X_test_s = sparse.dok_matrix((len(X_test), num_words+1), dtype=np.int8)

for row_id, word_freq_dict in X_test.items():
    for word_id, word_count in word_freq_dict.items():
        X_test_s[row_id, word_id] = word_count

X_test_sparse = X_test_s.tocsr()

# Fit, Predict and Basic Evaluation

In [33]:
softReg = LogisticRegression(multi_class = 'multinomial', solver='lbfgs')
softReg.fit(X_train_sparse, Y_train)

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(multi_class='multinomial')

In [34]:
Y_pred = softReg.predict(X_test_sparse)

In [35]:
print("Number of mislabeled points out of a total %d points : %d"
      % (len(Y_test), (Y_test != Y_pred).sum()))

Number of mislabeled points out of a total 3534 points : 1288


# +9% compared to Naive-Bayes

In [21]:
(3534-1288)/3524

0.6373439273552781